# Updater handler

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
from sweat.timeseries import status_handler as sh
from sweat.timeseries import updater_handler as uh
from sweat.timeseries.types import TimeSeriesVar as TSVar

## Create input data

In [ ]:
# Generate 7 dates until now
window = 7
today = pd.Timestamp.now()
dates = pd.date_range(end=today, periods=window, freq="D").date

# Create dimensions
size_x = 1200
size_y = 1000
x = np.arange(size_x)
y = np.arange(size_y)

shape = (size_y, size_x, window)

# Create data variables
et_data = np.full(shape, np.nan)
radiation_data = np.ones(shape)
flags_data = np.full(shape, sh.INIT_STATUS, dtype=sh.STATUS_TYPE)

# Create the dataset
ts = xr.Dataset(
    data_vars={
        TSVar.ET.value: (["y", "x", "time"], et_data),
        TSVar.RADIATION.value: (["y", "x", "time"], radiation_data),
        TSVar.FLAGS.value: (["y", "x", "time"], flags_data),
    },
    coords={
        "time": dates,
        "x": x,
        "y": y,
    },
)

In [ ]:
# Select first date randomly from first 4 dates (leaves room for gap)
first_idx = np.random.randint(0, 4)
# Select second date from at least 3 positions after first
second_idx = np.random.randint(first_idx + 3, len(dates))

acquisition_dates = sorted([dates[first_idx], dates[second_idx]])

# Create data variables
et_feed = np.random.uniform(low=3.0, high=5.5, size=(size_y, size_x, 2))
valid_feed = np.random.randint(0, 2, size=(size_y, size_x, 2))

# Create the dataset
feed = xr.Dataset(
    data_vars={
        TSVar.ET.value: (["y", "x", "time"], et_feed),
        TSVar.VALID.value: (["y", "x", "time"], valid_feed),
    },
    coords={
        "time": acquisition_dates,
        "x": x,
        "y": y,
    },
)

### Create a updater and update a time series

In [ ]:
config = {"method": "linear", "params": {"parallel": True, "num_workers": 4}}

In [ ]:
uh.UpdaterConfig.model_validate(config)

In [ ]:
updater_config = uh.UpdaterConfig.model_validate(config)

In [ ]:
updater_config

In [ ]:
updater = uh.create(method=updater_config.method, params=updater_config.params)

### Run 

In [ ]:
config = {"method": "linear", "params": {"parallel": True}}

In [ ]:
%time uh.run(ts, feed=feed, config=config)

In [ ]:
config = {"method": "linear", "params": {"parallel": False}}

In [ ]:
%time uh.run(ts, feed=feed, config=config)